# Augmentation by parapharsing

## Init & Load Seed Data

In [5]:
import json
from openai import OpenAI
from tqdm import tqdm 
import os


In [6]:
DOMAIN = "drone-planning/"
# DOMAIN = "clean-up/"
# DOMAIN = "pick-and-place/"
with open(DOMAIN + "train_seed.jsonl") as f:
    train_seed = [json.loads(line) for line in f]

In [7]:
eng_seeds = {
    seed['natural']: [] for seed in train_seed
}

## Augmentation Code
prompting GPT-3 seems to work the best in this case

In [35]:
# You need to set your OPENAI API key here
# https://beta.openai.com/account/api-keys

MODEL = "gpt-5.4" #"gpt-5.4"
client = OpenAI(api_key=os.environ.get("OPEN_AI_API_KEY"))

In [54]:
def normalize(sentence):
    # captialize first letter and add period at the end if not present
    if sentence[0].islower():
        sentence = sentence[0].upper() + sentence[1:]
    if sentence[-1] != '.':
        sentence = sentence + '.'
    return sentence

def parse_sentences_from_response(response):
    lines = response.split('\n')
    # assert len(lines) == 5
    assert len(lines) == 10
    # lines[0] = "1. " + lines[0]
    paraphrases = []
    for idx, line in enumerate(lines):
        assert line.startswith(str(idx+1) + '. ')
        sentence_start_idx = len(str(idx+1) + '. ')
        paraphrases.append(line[sentence_start_idx:])
    for paraphrase in paraphrases:
        if paraphrase[-1] == ' ':
            if paraphrase[-2] == '.':
                paraphrase = paraphrase[:-1]
            else:
                paraphrase = paraphrase[:-2] + '.'
    return paraphrases


PROMPT = """Rephrase the source sentence in 10 different ways. Make the outputs as diverse as possible.

Source: 
SOURCE-TO-BE-PLACED

Outputs:
1."""
def rephrase_a_sentence(sentence):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": PROMPT.replace("SOURCE-TO-BE-PLACED", normalize(sentence))}
        ],
        temperature=0.7,
        # max_output_tokens=512
    )
    output = response.choices[0].message.content
    # print(output)
    try:
        paraphrases = parse_sentences_from_response(output)
    except:
        print("Error in parsing response")
        print(output)
        return output, "ERROR"
    return paraphrases

In [44]:
O = rephrase_a_sentence("Go to the red room or go to the green room to finally go to the blue room.")

1. Head to the red room or the green room, and then make your way to the blue room.
2. First go into either the red room or the green room, then proceed to the blue room.
3. Visit the red room or choose the green room before ending up in the blue room.
4. You can enter the red room or the green room, but in the end, go to the blue room.
5. Make a stop in the red room or the green room, then continue on to the blue room.
6. Start by going to either the red room or the green room, and finish in the blue room.
7. Go through the red room or the green room before finally reaching the blue room.
8. Either head for the red room or the green room, and afterward go to the blue room.
9. Begin in the red room or the green room, then wrap up by going to the blue room.
10. Choose between the red room and the green room first, then go to the blue room.


In [45]:
O

['Head to the red room or the green room, and then make your way to the blue room.',
 'First go into either the red room or the green room, then proceed to the blue room.',
 'Visit the red room or choose the green room before ending up in the blue room.',
 'You can enter the red room or the green room, but in the end, go to the blue room.',
 'Make a stop in the red room or the green room, then continue on to the blue room.',
 'Start by going to either the red room or the green room, and finish in the blue room.',
 'Go through the red room or the green room before finally reaching the blue room.',
 'Either head for the red room or the green room, and afterward go to the blue room.',
 'Begin in the red room or the green room, then wrap up by going to the blue room.',
 'Choose between the red room and the green room first, then go to the blue room.']

## Run Augmentation

In [46]:
len(eng_seeds)

540

In [52]:
import random
rand_idx = random.randrange(0, len(eng_seeds)-1, 1)
list(eng_seeds.keys())[rand_idx]

'avoid go to the red room until go to the second floor'

In [55]:
def paraphrase_done(eng_seeds):
    for eng_seed, extended in tqdm(eng_seeds.items()):
        if len(extended) == 0:
            return False
    return True

while not paraphrase_done(eng_seeds):
    for eng_seed, extended in tqdm(eng_seeds.items()):
        if len(extended) == 0:
            extended += rephrase_a_sentence(eng_seed)

100%|██████████| 540/540 [00:00<00:00, 443580.92it/s]


In [56]:
eng_seeds

{'go to the blue room': ['Head to the blue room.  ',
  'Make your way to the blue room.  ',
  'Proceed to the blue room.  ',
  'Walk into the blue room.  ',
  'Enter the blue room.  ',
  'Move over to the blue room.  ',
  'Please go to the blue room.  ',
  'Kindly proceed to the blue room.  ',
  'Find your way to the blue room.  ',
  'Step into the blue room.'],
 'go to the blue room, and then the green room at last': ['Head to the blue room first, and finish in the green room.',
  'Visit the blue room, then end up in the green room.',
  'Start by going to the blue room, and finally proceed to the green room.',
  'Make your way to the blue room, and afterward go to the green room last.',
  'Go into the blue room before heading to the green room at the end.',
  'First enter the blue room, then conclude in the green room.',
  'Begin with the blue room and wrap up by going to the green room.',
  'Stop by the blue room, and afterward head to the green room as your final destination.',
  'G

### Dump as Training Data

In [57]:
train_seed[0]

{'canonical': 'finally ( the blue room )',
 'formula': 'finally ( the blue room )',
 'natural': 'go to the blue room'}

In [58]:
with open(DOMAIN + "syn-aug.train.jsonl", 'w') as f:
    for seed in train_seed:
        f.write(json.dumps(seed) + '\n')
        for aug_eng in eng_seeds[seed['natural']]:
                f.write(json.dumps({
                    'natural': aug_eng,
                    'canonical': seed['canonical'],
                    'formula': seed['formula']
                }) + '\n')

In [59]:
with open(DOMAIN + "syn.train.jsonl", 'w') as f:
    for seed in train_seed:
        f.write(json.dumps(seed) + '\n')

### Normalize the natural language form 

In [60]:
if DOMAIN == "clean-up/":
    # in clean up, golden natural language data comes without period at the end, no capitalization in the beginning
    def clean_up_normalize(sentence):
        if sentence[0].isupper():
            sentence = sentence[0].lower() + sentence[1:]
        if sentence[-1] == '.':
            sentence = sentence[:-1]
        return sentence

    buffer = []
    with open(DOMAIN + "syn-aug.train.jsonl", 'r') as f:
        for l in f.readlines():
            buffer.append(json.loads(l))
    
    with open(DOMAIN + "syn-aug.train.jsonl", 'w') as f:
        for dp in buffer:
            f.write(json.dumps({
                'natural': clean_up_normalize(dp['natural']),
                'canonical': dp['canonical'],
                'formula': dp['formula']
            }) + '\n')

if DOMAIN == "pick-and-place/":
    # in pick and place, golden natural language data comes without period at the end, no capitalization in the beginning
    def clean_up_normalize(sentence):
        if sentence[0].isupper():
            sentence = sentence[0].lower() + sentence[1:]
        if sentence[-1] == '.':
            sentence = sentence[:-1]
        return sentence

    buffer = []
    with open(DOMAIN + "syn-aug.train.jsonl", 'r') as f:
        for l in f.readlines():
            buffer.append(json.loads(l))
    
    with open(DOMAIN + "syn-aug.train.jsonl", 'w') as f:
        for dp in buffer:
            f.write(json.dumps({
                'natural': clean_up_normalize(dp['natural']),
                'canonical': dp['canonical'],
                'formula': dp['formula']
            }) + '\n')

In [61]:
if DOMAIN == "drone-planning/":
    # in clean up, golden natural language data comes with a "space + period" at the end, no capitalization in the beginning
    def clean_up_normalize(sentence):
        if sentence[0].isupper():
            sentence = sentence[0].lower() + sentence[1:]
        while sentence[-1] == ' ' or sentence[-1] == '.' or sentence[-1] == '!':
            sentence = sentence[:-1]
        sentence = sentence + '.'
        sentence = sentence.replace('.', ' .')
        sentence = sentence.replace(',', ' ,')
        return sentence

    buffer = []
    # with open(DOMAIN + "syn-aug.train.jsonl", 'r') as f:
    #     for l in f.readlines():
    #         buffer.append(json.loads(l))
    
    # with open(DOMAIN + "syn-aug.train.jsonl", 'w') as f:
    #     for dp in buffer:
    #         f.write(json.dumps({
    #             'natural': clean_up_normalize(dp['natural']),
    #             'canonical': dp['canonical'],
    #             'formula': dp['formula']
    #         }) + '\n')
    with open(DOMAIN + "syn.train.jsonl", 'r') as f:
        for l in f.readlines():
            buffer.append(json.loads(l))
    
    with open(DOMAIN + "syn.train.jsonl", 'w') as f:
        for dp in buffer:
            f.write(json.dumps({
                'natural': clean_up_normalize(dp['natural']),
                'canonical': dp['canonical'],
                'formula': dp['formula']
            }) + '\n')